In [1]:
# Cell 1 — Autoreload only
%load_ext autoreload
%autoreload 2

In [2]:
# Cell 1.5 - setup env
import sys
from pathlib import Path
sys.path.append(str(next(p for p in Path().resolve().parents if (p / "src").exists()) / "src"))

import setup_env
_ = setup_env.initialize_environment()

In [3]:
# Cell 2 — Imports
import re
import api_handler
import metrics
import cs_analysis
import sync

In [4]:
# Cell 3 — Environment + single puuid fetch
env = setup_env.initialize_environment()
repo_root = env["repo_root"]

SUMMONER_NAME = "RainbowThenga#420"
clean_name = re.sub(r'[^a-zA-Z0-9]', '_', SUMMONER_NAME)
BASE_DATA_PATH = repo_root / "data" / "users"
USER_DATA_PATH = BASE_DATA_PATH / clean_name

# Fetch puuid once — reuse everywhere
PUUID = api_handler.get_summoner_info(SUMMONER_NAME)["puuid"]

In [5]:
# Cell 4 — Sync
sync.sync_user_data(
    summoner_name=SUMMONER_NAME,
    base_data_path=BASE_DATA_PATH,
    champion_name=None,
    start_time=None,
    end_time=None,
)


📥 Syncing data for summoner: RainbowThenga#420
   Champion filter: None
🔍 Found 1 new matches to check.
→ Checking match EUW1_7748938705 for filters...
   Skipped (not Ranked Solo).
No new matches to download timelines for.
↪ Champion mastery already exists. Skipping.

✅ Sync complete.



In [6]:
# Cell 5 — Champion counts
metrics.champ_counts(summoner_riot_id=SUMMONER_NAME, repo_root=repo_root)

Counter({'Aphelios': 53,
         'Kalista': 6,
         'Senna': 2,
         'Rakan': 2,
         'Jhin': 1,
         'Kaisa': 1,
         'Lulu': 1,
         'Pantheon': 1})

In [7]:
# Cell 6 — Recent matchup summaries
metrics.summarize_recent_matchups(
    summoner_name=SUMMONER_NAME,
    user_data_path=USER_DATA_PATH,
    num_matches=10,
)

🧾 Last 10 matchups for RainbowThenga#420:

EUW1_7758498422:     You: Aphelios + Lux
EUW1_7758378725:     You: Aphelios + Nautilus
EUW1_7749151110:     You: Aphelios + Lulu
EUW1_7749048551:     You: Aphelios + Lux
EUW1_7748963999:     You: Aphelios + Nami
EUW1_7748025439:     You: Aphelios + Pyke
EUW1_7727707785:     You: Draven + Rakan
EUW1_7727694021:     You: Draven + Rakan
EUW1_7727620063:     You: Aphelios + Brand
EUW1_7726723595:     You: Aphelios + Brand


In [9]:
# Cell 7 — Single match summary
match_id = "EUW1_7727707785"
summary = metrics.summarize_match_for_notes(match_id, SUMMONER_NAME, USER_DATA_PATH)
print(summary)

## Match Summary: EUW1_7727707785

Champion: Rakan
KDA: 0/2/22
CS: 38 (1.0/min), CS@15: 29 (1.93/min)
Damage Dealt: 10,193 (Rank 10/10)

Bot Lane:

    You: Draven + Rakan

    Enemy: Tristana + Leona

Result: Win
Game Duration: 39 min

Tags: #leagueoflegends #support #champ-Rakan #ally-champ-Draven #opp-champ-Tristana #opp-champ-Leona #win-yes




In [ ]:
# Cell 8 — CS timeline (pass PUUID directly, no API call)
cs_curve = cs_analysis.extract_cs_timeline_from_files(match_id, PUUID, USER_DATA_PATH)
print(cs_curve)